In [32]:
from ultralytics import YOLO
import cv2
import numpy as np

In [33]:
model = YOLO('yolov8n.pt')

In [34]:
img_path = '../data/raw/8-2-second.jpg'
img = cv2.imread(img_path)

In [35]:
result = model(img, imgsz=960 , conf=0.15)[0]


0: 736x960 2 persons, 20 cars, 3 motorcycles, 1 bus, 1 train, 4 trucks, 86.0ms
Speed: 5.2ms preprocess, 86.0ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 960)


In [36]:
#car, motorcycle, bus, truck
VALID_CLASSES = {2, 3, 5, 7}

vehicles = []
for box in result.boxes:
    cls = int(box.cls[0])
    if cls in VALID_CLASSES:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        vehicles.append((x1, y1, x2, y2, cls))


In [37]:
vehicles.sort(key= lambda b: b[0])

widths = [x2 - x1 for (x1, y1, x2, y2, cls) in vehicles]
avg_width = np.mean(widths)

In [38]:
THRESHOLD = 1.2 * avg_width

In [39]:
for (x1, y1, x2, y2, cls) in vehicles:
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)

In [40]:
for i in range(len(vehicles) - 1):
        x1_a, y1_a, x2_a, y2_a, cls_a = vehicles[i]
        x1_b, y1_b, x2_b, y2_b, cls_b = vehicles[i + 1]

        gap = x1_b - x2_a

        if gap > THRESHOLD:
            # Draw the parking lot
            x_start = x2_a
            x_end = x1_b
            y_top = min(y1_a, y1_b)
            y_bottom = max(y2_a, y2_b)

            cv2.rectangle(img, (x_start, y_top), (x_end, y_bottom), (0, 255, 0), 2)
            cv2.putText(img, "Park Here", (x_start + 5, y_top - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [41]:
cv2.imwrite("../data/processed/resultado4.jpg", img)


True